## Práctica de Limpieza de Texto con Pandas

### Por:

Juan Pablo Zapata Zapata

### Fecha:

2025-10-24

### Descripción:

Este notebook implementa técnicas de limpieza y normalización de texto utilizando pandas. Se trabajan diferentes tipos de datos textuales como texto general, tweets, productos y datos personales, aplicando transformaciones como normalización de mayúsculas/minúsculas, eliminación de espacios, puntuación, acentos, emojis, y extracción de elementos específicos como emails, URLs, hashtags y menciones.


## 📚 Importar librerías


In [ ]:
# Librerías base para ciencia de datos
import pandas as pd
import numpy as np
import re

# Configuración de visualización
pd.set_option("display.max_colwidth", None)


: 

## 💾 Cargar datos


In [ ]:
## Conjuntos de datos de juguete
# Creamos 4 DataFrames para practicar distintas tareas de limpieza

## Texto general
df_texto = pd.DataFrame(
    {
        "texto": [
            "  ¡Hola Mundo!!!  Esto es   un EJEMPLO: visita https://miweb.com  #DataScience  :)  ",
            "Pandas > numpy? 🤔  Email: persona@example.com   ",
            "Me gusta el café colombiano; es buenísimo!!! #Café #Colombia @juan",
            "Oferta!!! 3x2 en jabón líquido 500ml - CÓDIGO: A-123",
            "      TABLAS\t, \n espacios   y saltos de línea.\r\n",
            "Teléfono: (57) 300-123-45-67; Whatsapp +57 300 222 33 44",
            "Dirección: Cll 10 # 5-20; Medellín. Barrio: La América",
            "Emoji test: 😀🙌🏽🏳️‍🌈, símbolos ©®™ y otros…",
        ]
    }
)

## Tweets
df_tweets = pd.DataFrame(
    {
        "tweet": [
            "RT @maria: Nuevo post en el blog -> http://blog.com/post?id=45 #nlp #python",
            "¡Me encanta Pandas! #datos #Python https://example.org @data_science 😊",
            "Probando cosas en Jupyter... sin link ni hashtag",
            "@juan y @ana lanzaron curso de NLP en https://cursos.ai #nlp #ml",
            "¿Pandas o Polars? debátanlo aquí 👉 https://foro.com #data",
        ]
    }
)

## Productos y precios
df_productos = pd.DataFrame(
    {
        "producto": [
            "Camisa talla M",
            "Pantalón-XL",
            "Zapato, Talla: 42",
            "Blusa s",
            "Medias 10-12",
            "Polo Talla l",
            "Vestido - 36",
            "Sombrero (talla Única)",
        ],
        "precio": [
            "$1.234,50",
            "USD 45",
            "30,00 €",
            "25.000",
            "$ 0",
            "S/. 120.90",
            "COP 9.990",
            "AR$ 2.550,00",
        ],
    }
)

## Personas, respuestas y direcciones
df_personas = pd.DataFrame(
    {
        "nombre": [
            "ana María LOPEZ",
            "Juan  perez",
            "Ñandú   Gómez",
            "Miguel (Soporte)",
            "  MÓNICA de la CRUZ  ",
            "luis-delgado",
        ],
        "categoria": ["Sí", "si", "SI ", "No", "—", None],
        "direccion": [
            "Calle 45 # 12-34, Bogotá",
            "Av. Siempre Viva 742 - Lima",
            "Cll. 10 No. 5-20 Medellín",
            "Cra 7a # 45-60, Bogotá",
            "Av. 9 #12-34  Cali",
            "Av. Insurgentes Sur 1234, CDMX",
        ],
        "id_raw": [
            "abc-0001",
            "abc 001",
            "ABC0002",
            "Abc_003",
            "abc-00004",
            "ABC-0005",
        ],
    }
)

print("df_texto:")
display(df_texto)
print("\ndf_tweets:")
display(df_tweets)
print("\ndf_productos:")
display(df_productos)
print("\ndf_personas:")
display(df_personas)


## 👷 Preparación de datos o Ingeniería de características


## Inspección rápida de datos


In [ ]:
# Exploración de la estructura de los DataFrames
print("=== INSPECCIÓN DE DATAFRAMES ===")
print("\n1. df_texto - Info:")
df_texto.info()
print("\n2. df_tweets - Info:")
df_tweets.info()
print("\n3. df_productos - Info:")
df_productos.info()
print("\n4. df_personas - Info:")
df_personas.info()

# Observaciones:
# - df_texto: Requiere limpieza de espacios, puntuación, acentos, emojis, URLs
# - df_tweets: Necesita extracción de hashtags, menciones, URLs y limpieza de RT
# - df_productos: Requiere estandarización de tallas y normalización de precios
# - df_personas: Necesita limpieza de nombres, categorías y direcciones


### 1) Normalización a minúsculas


In [ ]:
df_texto["texto_min"] = df_texto["texto"].astype(str).str.lower()
df_texto[["texto", "texto_min"]]


### 2) Eliminación de espacios


In [ ]:
df_texto["texto_espacios"] = df_texto["texto_min"].str.strip()
df_texto[["texto_espacios", "texto_min"]]


### 3) Eliminación de puntuación


In [ ]:
df_texto["texto_sin_punct"] = df_texto["texto_espacios"].str.replace(
    r"[^\w\s]", "", regex=True
)
df_texto["texto_sin_punct"]


### 4) Eliminación de acentos


In [ ]:
def quitar_acentos(text: str) -> str:
    MAP_VOCALES = {
        "á": "a",
        "é": "e",
        "í": "i",
        "ó": "o",
        "ú": "u",
        "ü": "u",
    }
    translate = str.maketrans(MAP_VOCALES)
    text = text.translate(translate)
    return text


df_texto["texto_sin_acentos"] = df_texto["texto_sin_punct"].apply(quitar_acentos)
df_texto["texto_sin_acentos"]


### 5) Eliminación de emojis


In [ ]:
emoji_re = re.compile("[\U0001f300-\U0001faff\U00002700-\U000027bf]+", flags=re.UNICODE)
df_texto["texto_sin_emoji"] = df_texto["texto_espacios"].apply(
    lambda s: emoji_re.sub("", s)
)
df_texto["texto_sin_emoji"]


### 6) Extracción de emails y URLs


In [ ]:
email_pat = r"[A-Za-z0-9_.+-]+@[A-Za-z0-9-]+\.[A-Za-z0-9.-]+"
url_pat = r"https?://\S+"
df_texto["email"] = df_texto["texto"].str.findall(email_pat)
df_texto["urls"] = df_texto["texto"].str.findall(url_pat)

df_texto[["texto", "email", "urls"]]


### 7) Hashtags, menciones y URLs en tweets


In [ ]:
pat_hash = r"#\w+"
pat_ment = r"@\w+"
pat_url = r"https?://\S+"
df_tweets["hashtags"] = df_tweets["tweet"].str.findall(pat_hash)
df_tweets["mentions"] = df_tweets["tweet"].str.findall(pat_ment)
df_tweets["urls"] = df_tweets["tweet"].str.findall(pat_url)

df_tweets


### 8) Detección y limpieza de retuits


In [ ]:
df_tweets["es_rt"] = df_tweets["tweet"].str.startswith("RT ")
df_tweets["tweet_sin_rt"] = df_tweets["tweet"].str.replace(
    r"^RT\s+@\w+:\s*", "", regex=True
)

df_tweets


### 9) Limpieza completa de tweets


In [ ]:
tmp = df_tweets["tweet_sin_rt"] if "tweet_sin_rt" in df_tweets else df_tweets["tweet"]
tmp = tmp.str.replace(r"https?://\S+", " ", regex=True)
tmp = tmp.str.replace(r"@\w+|#\w+", " ", regex=True)
tmp = tmp.str.replace(r"[^\w\sáéíóúÁÉÍÓÚñÑ]", " ", regex=True)
tmp = tmp.str.replace(r"\s+", " ", regex=True).str.strip().str.casefold()
df_tweets["tweet_limpio"] = tmp
df_tweets[["tweet", "tweet_limpio"]]


### 10) Extracción y estandarización de tallas


In [ ]:
pat_letra = r"(?i)\b(xs|s|m|l|xl|xxl|única|unica|u)\b"
pat_num = r"(\d+(?:\.\d+)?)"


def extraer_talla(row):
    txt = str(row["producto"])
    m1 = re.search(pat_letra, txt)
    m2 = re.search(pat_num, txt)
    talla_std = None
    talla_num = None
    if m1:
        val = m1.group(1).lower()
        talla_std = "TU" if val in {"única", "unica", "u"} else val.upper()
    if m2:
        talla_num = float(m2.group(1)) if m2.group(1) is not None else None
    return pd.Series({"talla_std": talla_std, "talla_num": talla_num})


df_productos[["talla_std", "talla_num"]] = df_productos.apply(extraer_talla, axis=1)
df_productos


### 11) Normalización de precios


In [ ]:
def parse_precio(s):
    if pd.isna(s):
        return np.nan
    s = str(s).replace("\xa0", " ").strip()
    s2 = re.sub(r"[^0-9,.-]", "", s)
    if "," in s2 and "." in s2:
        s2 = s2.replace(".", "").replace(",", ".")
    elif "," in s2 and "." not in s2:
        s2 = s2.replace(",", ".")
    s2 = s2.replace(",", "")
    try:
        return float(s2)
    except Exception:
        return np.nan


df_productos["precio_num"] = df_productos["precio"].apply(parse_precio)
df_productos


### 12) Filtrado por texto


In [ ]:
mask = (
    df_productos["producto"]
    .str.lower()
    .apply(quitar_acentos)
    .str.contains("camisa|pantalon")
)
df_productos[mask]


### 13) Limpieza de nombres propios


In [ ]:
def limpiar_nombre(s):
    s = re.sub(r"\([^)]*\)", " ", str(s))
    s = s.replace("-", " ")
    s = re.sub(r"\s+", " ", s).strip()
    s = s.title()
    for w in [" De ", " Del ", " La ", " Y "]:
        s = s.replace(w, w.lower())
    return s


df_personas["nombre_limpio"] = df_personas["nombre"].apply(limpiar_nombre)
df_personas["nombre_limpio"]


### 14) Normalización de respuestas categóricas


In [ ]:
def a_bool_si(s):
    if s is None:
        return False
    val = str(s).strip().casefold()
    return val in {"si", "sí", "si.", "sí."}


df_personas["categoria_bool"] = df_personas["categoria"].apply(a_bool_si)
df_personas


### 15) Extracción de ciudad desde dirección


In [ ]:
def extraer_ciudad(s):
    s = str(s)
    m = re.search(r",\s*([^,]+)$", s)
    if m:
        return m.group(1).strip()
    return s.strip().split()[-1]


df_personas["ciudad"] = df_personas["direccion"].apply(extraer_ciudad)
df_personas


### 16) Normalización y validación de IDs


In [ ]:
def normalizar_id(s):
    m = re.search(r"([A-Za-z]{3})\s*[-_ ]?\s*(\d+)", str(s))
    if not m:
        return None
    pref = m.group(1).upper()
    num = int(m.group(2))
    return f"{pref}-{num:04d}"


df_personas["id_norm"] = df_personas["id_raw"].apply(normalizar_id)
df_personas["id_valido"] = df_personas["id_norm"].str.match(r"^[A-Z]{3}-\d{4}$")
df_personas[["id_raw", "id_norm", "id_valido"]]


### 17) Pipeline de limpieza general


In [ ]:
def limpia_basica(s: str) -> str:
    """Limpieza básica de texto.

    Convierte a minúsculas, quita acentos, URLs, menciones y hashtags,
    puntuación, dígitos y quita espacios.

    Arguments:
        s (str): Texto a limpiar.

    Returns:
        str: Texto limpio.

        Ejemplo:
        >>> limpia_basica('  ¡Hola Mundo!!! Visita https://miweb.com  #DataScience  :)  ')
        'hola mundo visita'
    """
    if s is None:
        return s
    s = str(s).lower()
    MAP_VOCALES = {
        "á": "a",
        "é": "e",
        "í": "i",
        "ó": "o",
        "ú": "u",
        "ü": "u",
    }
    translate = str.maketrans(MAP_VOCALES)
    s = s.translate(translate)
    s = re.sub(r"https?://\S+", " ", s)
    s = re.sub(r"[@#]\w+", " ", s)
    s = re.sub(r"[^a-z\sñ]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


df_texto["texto_limpio_final"] = df_texto["texto"].apply(limpia_basica)
df_texto[["texto", "texto_limpio_final"]]


### 18) Análisis de cambio en longitud del texto


In [ ]:
df_texto["len_raw"] = df_texto["texto"].str.len()
df_texto["len_clean"] = df_texto["texto_limpio_final"].str.len()
df_texto[["texto", "texto_limpio_final", "len_raw", "len_clean"]]


## 📊 Análisis de Resultados y Conclusiones

El análisis de los resultados muestra la efectividad de las técnicas de limpieza de texto aplicadas:

1. Reducción significativa de longitud: En todos los casos, la limpieza redujo considerablemente la longitud del texto, eliminando elementos no informativos como espacios extra, puntuación, emojis y URLs.

2. Estandarización exitosa: Se logró normalizar diferentes formatos de datos:

- Precios en múltiples monedas y formatos

- Tallas de productos en sistemas numéricos y de letras

- Nombres propios con diferentes formatos

- Identificadores con variaciones en el formato

3. Extracción efectiva de elementos: Las expresiones regulares demostraron ser efectivas para extraer emails, URLs, hashtags y menciones de manera precisa.

4. Manejo de caracteres especiales: La limpieza de acentos, emojis y caracteres especiales permitió obtener texto más adecuado para procesamiento posterior.

La limpieza de texto es fundamental como paso previo a cualquier análisis de NLP o procesamiento de datos, ya que mejora la calidad y consistencia de los datos.


## 💡 Propuestas e Ideas

1. Automatización de pipelines: Crear pipelines reutilizables para diferentes tipos de datos textuales.

2. Validación de datos: Implementar checks de calidad para verificar que la limpieza no elimine información importante.

3. Escalabilidad: Adaptar las funciones para trabajar con grandes volúmenes de datos usando procesamiento paralelo.

4. Internacionalización: Extender las funciones para soportar más idiomas y conjuntos de caracteres.

5. Interfaz de usuario: Desarrollar una herramienta interactiva para que usuarios no técnicos puedan aplicar estas limpiezas.


## 📖 Referencias

- Procesamiento básico de texto (curso de NLP): https://joserzapata.github.io/courses/nlp/procesamiento-basico/

- Documentación de pandas.Series.str: https://pandas.pydata.org/docs/reference/series.html#string-handling

- Expresiones regulares en Python: https://docs.python.org/3/library/re.html
